# 03-3. 해석 가능한 특징 기반 지도·비지도 분석

정상 금융상담과 보이스피싱에 동일한 규칙을 적용하여 금전요구·개인정보요구·강압·긴급성·위협·비밀유지 등의 특징을 추출합니다.

- 지도학습: Logistic Regression, Linear SVM, Random Forest, Extra Trees 비교
- 비지도학습: K-means로 정상·사기 혼합 군집 및 보이스피싱 내부 전술 군집 탐색
- 원문 길이·출처·파일명·금융주제는 모델 입력에서 제외
- 횟수 대신 1,000자당 표현 밀도와 존재 여부 사용


In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow scikit-learn scipy seaborn matplotlib koreanize-matplotlib joblib openpyxl

In [ ]:
# 1. 라이브러리와 Google Drive
from google.colab import drive
from pathlib import Path
from IPython.display import display
import json, re, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score, average_precision_score,
    classification_report, confusion_matrix, normalized_mutual_info_score,
    precision_recall_fscore_support, silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
warnings.filterwarnings('ignore')
drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 설정값

In [ ]:
# 2. 경로와 공통 설정
DRIVE_ROOT=Path('/content/drive/MyDrive')
PROJECT_ROOT=DRIVE_ROOT/'보이스피싱_분석'
DATASET_ROOT=PROJECT_ROOT/'구축 데이터셋_v3'
ML_ROOT=DATASET_ROOT/'02_ml_tables'
OUTPUT_ROOT=PROJECT_ROOT/'해석형_특징분석결과_v1'
TABLE_ROOT=OUTPUT_ROOT/'01_분석표'
FIGURE_ROOT=OUTPUT_ROOT/'02_그래프'
MODEL_ROOT=OUTPUT_ROOT/'03_모델'
PRED_ROOT=OUTPUT_ROOT/'04_예측_군집결과'
REPORT_ROOT=OUTPUT_ROOT/'05_보고서'
for folder in [TABLE_ROOT,FIGURE_ROOT,MODEL_ROOT,PRED_ROOT,REPORT_ROOT]: folder.mkdir(parents=True,exist_ok=True)
SEED=42
TEST_RATIO=.20
DEV_RATIO=.20
FRAUD='VOICE_PHISHING'
NORMAL='LEGITIMATE_FINANCIAL_CALL'
assert ML_ROOT.exists(),f'구축 데이터셋_v3 경로를 확인하세요: {ML_ROOT}'
print('입력:',DATASET_ROOT); print('출력:',OUTPUT_ROOT)

## 2. 데이터 불러오기

In [ ]:
# 3. 정상·사기와 사기유형 테이블
def read_table(folder,name):
    pq=folder/f'{name}.parquet'; csv=folder/f'{name}.csv'
    if pq.exists(): return pd.read_parquet(pq)
    assert csv.exists(),f'{name}을 찾지 못했습니다.'
    return pd.read_csv(csv,encoding='utf-8-sig')
det=read_table(ML_ROOT,'fraud_detection_ml')
type_df=read_table(ML_ROOT,'fraud_type_ml')
required={'conversation_id','group_id','fraud_label','model_input_text','original_split'}
assert required.issubset(det.columns),f'필수 컬럼 누락: {required-set(det.columns)}'
det=det.dropna(subset=['group_id','fraud_label','model_input_text']).copy()
det['group_id']=det['group_id'].astype(str)
display(det.fraud_label.value_counts().rename_axis('구분').reset_index(name='건수'))
print('분석 대상:',len(det),'건')

## 3. 동일 규칙으로 해석형 특징 추출

아래 규칙은 정답 라벨을 보지 않고 정상상담과 보이스피싱 전체에 동일하게 적용합니다. 자동 추출 결과이므로 `SILVER` 특징입니다.

In [ ]:
# 4. 행동·심리·말투 특징 규칙
FEATURE_RULES={
 'money_request':r'송금|이체|입금|납부|지불|결제|돈.{0,8}(보내|내|줘|준비)|금액.{0,8}(보내|입금)',
 'transfer_cash':r'계좌.{0,8}(이체|송금)|현금.{0,8}(인출|찾|전달)|ATM|씨디기|CD기',
 'fee_tax_deposit':r'수수료|선입금|보증금|예치금|공탁금|세금|과태료|벌금|인지대',
 'account_info':r'계좌번호|잔액|통장|카드번호|금융거래|거래내역',
 'personal_info':r'주민번호|주민등록|생년월일|신분증|주소|개인정보|명의',
 'auth_code':r'인증번호|비밀번호|보안카드|OTP|일회용.{0,3}비밀번호',
 'app_remote':r'앱.{0,8}(설치|깔)|어플.{0,8}(설치|깔)|원격.{0,8}(접속|제어)|팀뷰어|퀵서포트',
 'command_pressure':r'하세요|하셔야|해야 합니다|따라 하|지금.{0,8}(가|하|보내|이체)|시키는 대로|말씀드린 대로',
 'urgency_pressure':r'지금 당장|즉시|긴급|오늘 안|시간이 없|빨리|지체하면|마감|몇 분 안',
 'fear_threat':r'체포|구속|압류|범죄|수배|처벌|고소|고발|피해를 입|큰일|위험|납치',
 'isolation_secrecy':r'비밀|말하지 마|알리면 안|누구에게도|혼자만|통화.{0,8}(끊지|유지)|전화.{0,8}(끊지|받지)',
 'authority_trust':r'검찰|검사|경찰|수사관|법원|금융감독원|금감원|은행 본점|정부기관|공문|사건번호',
 'resistance_handling':r'의심|못 믿|확인해 보|그게 아니라|걱정하지|안심|오해|설명드리',
 'benefit_offer':r'대출.{0,10}(승인|가능|해드리)|환급|돌려드리|지원금|혜택|저금리|금리.{0,8}(낮|인하)|한도.{0,8}(상향|증액)',
}
FEATURE_KO={
 'money_request':'금전·송금요구','transfer_cash':'송금·현금행동','fee_tax_deposit':'수수료·세금·보증금',
 'account_info':'계좌정보 관련표현','personal_info':'개인정보 관련표현','auth_code':'인증정보 관련표현',
 'app_remote':'앱설치·원격접속','command_pressure':'명령·강압','urgency_pressure':'긴급성·시간압박',
 'fear_threat':'공포·위협','isolation_secrecy':'고립·비밀유지','authority_trust':'권위·신뢰형성',
 'resistance_handling':'의심·저항대응','benefit_offer':'이익·혜택제안'}
compiled_rules={name:re.compile(pattern,re.I) for name,pattern in FEATURE_RULES.items()}
print('추출 특징:',len(compiled_rules),'개')

In [ ]:
# 5. 통화 길이 대신 1,000자당 표현 밀도와 존재 여부 생성
def extract_features(text):
    text=re.sub(r'\s+',' ',str(text or '')).strip()
    denominator=max(len(text),1)
    row={}
    present=0
    for name,pattern in compiled_rules.items():
        count=len(pattern.findall(text))
        row[f'{name}_rate_1k']=count/denominator*1000
        row[f'{name}_flag']=int(count>0)
        present+=int(count>0)
    row['risk_feature_diversity_ratio']=present/len(compiled_rules)
    return row
feature_values=pd.DataFrame(det.model_input_text.map(extract_features).tolist(),index=det.index)
feature_df=pd.concat([det[['conversation_id','group_id','fraud_label','original_split']].copy(),feature_values],axis=1)
feature_cols=list(feature_values.columns)
assert not feature_values.isna().any().any()
assert not any(x in feature_cols for x in ['text_length','source_group','financial_topic','original_split'])
feature_df.to_csv(TABLE_ROOT/'해석형_특징_데이터.csv',index=False,encoding='utf-8-sig')
display(feature_df.head()); print('모델 입력 특징:',len(feature_cols),'개')

## 4. 특징별 정상·사기 차이와 통계 검정

In [ ]:
# 6. 밀도 중앙값·존재율·Mann-Whitney 검정
def bh_fdr(p_values):
    p=np.asarray(p_values,float); order=np.argsort(p); ranked=p[order]*len(p)/(np.arange(len(p))+1)
    ranked=np.minimum.accumulate(ranked[::-1])[::-1]; result=np.empty_like(ranked); result[order]=np.clip(ranked,0,1); return result
stats=[]
for name in FEATURE_RULES:
    rate=f'{name}_rate_1k'; flag=f'{name}_flag'
    normal=feature_df.loc[feature_df.fraud_label.eq(NORMAL),rate]
    fraud=feature_df.loc[feature_df.fraud_label.eq(FRAUD),rate]
    _,p=mannwhitneyu(fraud,normal,alternative='two-sided')
    flag_table=pd.crosstab(feature_df.fraud_label,feature_df[flag])
    flag_p=chi2_contingency(flag_table)[1] if flag_table.shape==(2,2) else 1.0
    stats.append({'특징코드':name,'특징':FEATURE_KO[name],
      '정상_1000자당평균':normal.mean(),'사기_1000자당평균':fraud.mean(),
      '정상_존재율':feature_df.loc[feature_df.fraud_label.eq(NORMAL),flag].mean(),
      '사기_존재율':feature_df.loc[feature_df.fraud_label.eq(FRAUD),flag].mean(),
      '평균밀도차이':fraud.mean()-normal.mean(),'밀도_p_value':p,'존재여부_카이제곱_p_value':flag_p})
stats_df=pd.DataFrame(stats)
stats_df['밀도_fdr_p_value']=bh_fdr(stats_df['밀도_p_value'])
stats_df['존재여부_카이제곱_fdr_p_value']=bh_fdr(stats_df['존재여부_카이제곱_p_value'])
stats_df=stats_df.sort_values('평균밀도차이',ascending=False)
display(stats_df); stats_df.to_csv(TABLE_ROOT/'특징별_정상사기_차이.csv',index=False,encoding='utf-8-sig')
plot_df=stats_df.sort_values('평균밀도차이')
plt.figure(figsize=(10,7)); sns.barplot(data=plot_df,y='특징',x='평균밀도차이',color='#377eb8')
plt.axvline(0,color='black',lw=1); plt.xlabel('사기 - 정상: 1,000자당 평균 출현 차이'); plt.title('행동·심리 표현 밀도 차이')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'01_특징밀도_정상사기차이.png',dpi=170); plt.show()

## 5. 원본 통화 단위 학습·검증·테스트 분리

In [ ]:
# 7. 정상상담 공식 Validation 보존, 보이스피싱은 원본 file_id 그룹 분리
def random_group_map(groups):
    groups=np.array(sorted(pd.Series(groups).dropna().astype(str).unique()))
    train_dev,test=train_test_split(groups,test_size=TEST_RATIO,random_state=SEED)
    train,dev=train_test_split(train_dev,test_size=DEV_RATIO,random_state=SEED)
    out={g:'TRAIN' for g in train}; out.update({g:'DEV' for g in dev}); out.update({g:'TEST' for g in test}); return out
feature_df['ml_split']=''
normal_mask=feature_df.fraud_label.eq(NORMAL)
official=feature_df.original_split.fillna('').astype(str).str.upper()
normal_test=normal_mask & official.str.contains('VALID|TEST')
if normal_test.any():
    normal_pool=normal_mask & ~normal_test
    groups=feature_df.loc[normal_pool,'group_id'].unique(); tr_group,dv_group=train_test_split(groups,test_size=DEV_RATIO,random_state=SEED)
    feature_df.loc[normal_pool & feature_df.group_id.isin(tr_group),'ml_split']='TRAIN'
    feature_df.loc[normal_pool & feature_df.group_id.isin(dv_group),'ml_split']='DEV'
    feature_df.loc[normal_test,'ml_split']='TEST'
else:
    normal_map=random_group_map(feature_df.loc[normal_mask,'group_id']); feature_df.loc[normal_mask,'ml_split']=feature_df.loc[normal_mask,'group_id'].map(normal_map)
fraud_map=random_group_map(feature_df.loc[~normal_mask,'group_id']); feature_df.loc[~normal_mask,'ml_split']=feature_df.loc[~normal_mask,'group_id'].map(fraud_map)
assert not feature_df.ml_split.eq('').any()
assert feature_df.groupby('group_id').ml_split.nunique().max()==1,'원본 통화 누출이 있습니다.'
split_summary=feature_df.groupby(['ml_split','fraud_label']).size().reset_index(name='건수')
display(split_summary); split_summary.to_csv(TABLE_ROOT/'학습검증테스트_분포.csv',index=False,encoding='utf-8-sig')

## 6. 해석형 특징 지도학습

In [ ]:
# 8. 후보 알고리즘
def candidates():
    return {
     'Dummy':DummyClassifier(strategy='prior'),
     'Logistic_Regression':Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED))]),
     'Linear_SVM':Pipeline([('scale',StandardScaler()),('model',LinearSVC(class_weight='balanced',random_state=SEED))]),
     'Random_Forest':RandomForestClassifier(n_estimators=500,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1),
     'Extra_Trees':ExtraTreesClassifier(n_estimators=500,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1)}
def score_values(model,x):
    idx=list(model.classes_).index(FRAUD)
    if hasattr(model,'predict_proba'): return model.predict_proba(x)[:,idx]
    score=model.decision_function(x); return score if idx==1 else -score
def metrics(y,pred,score):
    p,r,f1,_=precision_recall_fscore_support(y,pred,average='binary',pos_label=FRAUD,zero_division=0)
    return {'accuracy':accuracy_score(y,pred),'precision':p,'recall':r,'f1':f1,
            'pr_auc':average_precision_score((np.asarray(y)==FRAUD).astype(int),score)}
train=feature_df[feature_df.ml_split.eq('TRAIN')]; dev=feature_df[feature_df.ml_split.eq('DEV')]; test=feature_df[feature_df.ml_split.eq('TEST')]
compare_rows=[]
for name,model in candidates().items():
    model.fit(train[feature_cols],train.fraud_label); pred=model.predict(dev[feature_cols]); score=score_values(model,dev[feature_cols])
    compare_rows.append({'모델':name,**metrics(dev.fraud_label,pred,score)}); print(name,'완료')
compare_df=pd.DataFrame(compare_rows).sort_values(['pr_auc','f1'],ascending=False).reset_index(drop=True)
display(compare_df); compare_df.to_csv(TABLE_ROOT/'지도학습_모델비교_검증.csv',index=False,encoding='utf-8-sig')

In [ ]:
# 9. 선정 모델 최종 테스트 한 번
best_name=compare_df.iloc[0]['모델']; best_model=clone(candidates()[best_name])
train_dev=feature_df[feature_df.ml_split.isin(['TRAIN','DEV'])]
best_model.fit(train_dev[feature_cols],train_dev.fraud_label)
test_pred=best_model.predict(test[feature_cols]); test_score=score_values(best_model,test[feature_cols])
test_metrics=metrics(test.fraud_label,test_pred,test_score)
display(pd.DataFrame([{'모델':best_name,**test_metrics}]))
print(classification_report(test.fraud_label,test_pred,zero_division=0))
pred_df=test[['conversation_id','group_id','fraud_label']].copy(); pred_df['예측']=test_pred; pred_df['보이스피싱점수']=test_score; pred_df['정답여부']=pred_df.fraud_label.eq(test_pred)
pred_df.to_csv(PRED_ROOT/'해석형특징_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
joblib.dump({'model':best_model,'feature_columns':feature_cols,'feature_rules':FEATURE_RULES},MODEL_ROOT/'interpretable_fraud_model.joblib')
cm=confusion_matrix(test.fraud_label,test_pred,labels=[NORMAL,FRAUD])
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['정상','보이스피싱'],yticklabels=['정상','보이스피싱'])
plt.xlabel('예측'); plt.ylabel('실제'); plt.title('해석형 특징 모델 최종 테스트'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'02_해석형모델_혼동행렬.png',dpi=170); plt.show()

In [ ]:
# 10. 최종 모델 변수 중요도
estimator=best_model.named_steps['model'] if isinstance(best_model,Pipeline) else best_model
if hasattr(estimator,'coef_'):
    importance=np.asarray(estimator.coef_).ravel()
elif hasattr(estimator,'feature_importances_'):
    importance=np.asarray(estimator.feature_importances_)
else:
    importance=np.zeros(len(feature_cols))
importance_df=pd.DataFrame({'특징코드':feature_cols,'중요도':importance})
importance_df['특징']=importance_df.특징코드.map(lambda x:FEATURE_KO.get(x.replace('_rate_1k','').replace('_flag',''),x))
importance_df['형태']=np.where(importance_df.특징코드.str.endswith('_flag'),'존재여부',np.where(importance_df.특징코드.str.endswith('_rate_1k'),'1,000자당밀도','다양성'))
importance_df['절대중요도']=importance_df.중요도.abs(); importance_df=importance_df.sort_values('절대중요도',ascending=False)
display(importance_df.head(20)); importance_df.to_csv(TABLE_ROOT/'최종모델_변수중요도.csv',index=False,encoding='utf-8-sig')
top=importance_df.head(20).sort_values('중요도')
plt.figure(figsize=(10,8)); sns.barplot(data=top,y='특징코드',x='중요도',color='#4c72b0')
plt.axvline(0,color='black',lw=1); plt.title(f'{best_name} 변수 중요도'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'03_최종모델_변수중요도.png',dpi=170); plt.show()

## 7. 비지도 K-means: 정상·사기 혼합 군집

K-means는 정상·사기 정답을 보지 않고 특징이 비슷한 통화를 묶습니다. 군집 생성 후에만 정답과의 대응을 확인합니다.

In [ ]:
# 11. 다수 클래스 영향을 줄이기 위한 1:1 표본
fraud_part=feature_df[feature_df.fraud_label.eq(FRAUD)]
normal_part=feature_df[feature_df.fraud_label.eq(NORMAL)].sample(n=len(fraud_part),random_state=SEED)
cluster_input=pd.concat([fraud_part,normal_part]).sample(frac=1,random_state=SEED).reset_index(drop=True)
scaler=StandardScaler(); cluster_x=scaler.fit_transform(cluster_input[feature_cols])
cluster_rows=[]; cluster_models={}
for k in range(2,9):
    model=KMeans(n_clusters=k,n_init=30,random_state=SEED); label=model.fit_predict(cluster_x)
    model2=KMeans(n_clusters=k,n_init=30,random_state=SEED+1); label2=model2.fit_predict(cluster_x)
    cluster_rows.append({'군집수':k,'silhouette':silhouette_score(cluster_x,label),'seed_stability_ari':adjusted_rand_score(label,label2)})
    cluster_models[k]=(model,label)
k_compare=pd.DataFrame(cluster_rows).sort_values(['silhouette','seed_stability_ari'],ascending=False)
display(k_compare); k_compare.to_csv(TABLE_ROOT/'KMeans_정상사기_군집수비교.csv',index=False,encoding='utf-8-sig')
best_k=int(k_compare.iloc[0].군집수); kmeans,mixed_labels=cluster_models[best_k]
cluster_input['군집ID']=mixed_labels
alignment=pd.crosstab(cluster_input.군집ID,cluster_input.fraud_label,margins=True)
ari=adjusted_rand_score(cluster_input.fraud_label,mixed_labels); nmi=normalized_mutual_info_score(cluster_input.fraud_label,mixed_labels)
display(alignment); print('정답과 군집의 ARI:',round(ari,4),'NMI:',round(nmi,4))
alignment.to_csv(TABLE_ROOT/'KMeans_군집_정상사기대응.csv',encoding='utf-8-sig')
cluster_input.to_csv(PRED_ROOT/'KMeans_정상사기_군집결과.csv',index=False,encoding='utf-8-sig')

In [ ]:
# 12. 군집 시각화와 군집별 특징 프로필
pca=PCA(n_components=2,random_state=SEED); xy=pca.fit_transform(cluster_x)
plot_df=pd.DataFrame({'PCA1':xy[:,0],'PCA2':xy[:,1],'군집ID':mixed_labels,'실제구분':cluster_input.fraud_label})
fig,axes=plt.subplots(1,2,figsize=(15,6))
sns.scatterplot(data=plot_df,x='PCA1',y='PCA2',hue='군집ID',palette='tab10',alpha=.6,ax=axes[0]); axes[0].set_title('정답을 사용하지 않은 K-means 군집')
sns.scatterplot(data=plot_df,x='PCA1',y='PCA2',hue='실제구분',alpha=.6,ax=axes[1]); axes[1].set_title('같은 좌표의 실제 정상·사기')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'04_KMeans_정상사기_군집.png',dpi=170); plt.show()
profile=cluster_input.groupby('군집ID')[feature_cols].mean().T
profile.to_csv(TABLE_ROOT/'KMeans_군집별_특징프로필.csv',encoding='utf-8-sig')
display(profile.head(30))
joblib.dump({'scaler':scaler,'model':kmeans,'feature_columns':feature_cols},MODEL_ROOT/'kmeans_mixed_calls.joblib')

## 8. 비지도 K-means: 보이스피싱 내부 전술 유형

In [ ]:
# 13. 보이스피싱만 사용하여 행동·심리 패턴이 유사한 사건 묶기
fraud_cluster=feature_df[feature_df.fraud_label.eq(FRAUD)].copy().reset_index(drop=True)
fraud_x=StandardScaler().fit_transform(fraud_cluster[feature_cols])
rows=[]; models={}
for k in range(2,7):
    model=KMeans(n_clusters=k,n_init=30,random_state=SEED); labels=model.fit_predict(fraud_x)
    other=KMeans(n_clusters=k,n_init=30,random_state=SEED+1).fit_predict(fraud_x)
    rows.append({'군집수':k,'silhouette':silhouette_score(fraud_x,labels),'seed_stability_ari':adjusted_rand_score(labels,other)})
    models[k]=(model,labels)
fraud_k_compare=pd.DataFrame(rows).sort_values(['silhouette','seed_stability_ari'],ascending=False)
best_fraud_k=int(fraud_k_compare.iloc[0].군집수); fraud_model,fraud_labels=models[best_fraud_k]
fraud_cluster['전술군집ID']=fraud_labels
fraud_cluster['case_id']=fraud_cluster.conversation_id.str.replace(r'^fraud_','',regex=True)
type_map=type_df[['case_id','supervised_target']].drop_duplicates('case_id')
fraud_cluster=fraud_cluster.merge(type_map,on='case_id',how='left')
type_alignment=pd.crosstab(fraud_cluster.전술군집ID,fraud_cluster.supervised_target.fillna('MIXED_UNKNOWN'),margins=True)
profile=fraud_cluster.groupby('전술군집ID')[feature_cols].mean().T
display(fraud_k_compare); display(type_alignment); display(profile)
fraud_k_compare.to_csv(TABLE_ROOT/'KMeans_보이스피싱_군집수비교.csv',index=False,encoding='utf-8-sig')
type_alignment.to_csv(TABLE_ROOT/'KMeans_전술군집_기존유형대응.csv',encoding='utf-8-sig')
profile.to_csv(TABLE_ROOT/'KMeans_전술군집_특징프로필.csv',encoding='utf-8-sig')
fraud_cluster.to_csv(PRED_ROOT/'KMeans_보이스피싱_전술군집결과.csv',index=False,encoding='utf-8-sig')
joblib.dump({'model':fraud_model,'feature_columns':feature_cols},MODEL_ROOT/'kmeans_fraud_tactics.joblib')

## 9. 결과 요약과 한계

In [ ]:
# 14. 자동 보고서와 실행 기록
top_features=importance_df.head(8).특징코드.tolist()
report=[
 '# 03-3 해석형 특징 기반 분석 결과','', '## 지도학습','',
 f'- 최종 모델: {best_name}',f"- 최종 F1: {test_metrics['f1']:.4f}",f"- 최종 Recall: {test_metrics['recall']:.4f}",
 f"- 최종 PR-AUC: {test_metrics['pr_auc']:.4f}",f"- 주요 특징: {', '.join(top_features)}",'',
 '## 비지도학습','',f'- 정상·사기 혼합 K-means 최적 k: {best_k}',f'- 정답과 군집의 ARI: {ari:.4f}',f'- 정답과 군집의 NMI: {nmi:.4f}',
 f'- 보이스피싱 내부 전술 군집수: {best_fraud_k}','',
 '## 해석 시 주의','',
 '- 모든 특징은 정상·사기에 동일한 규칙을 적용했지만 사람이 확정한 GOLD가 아닌 SILVER 특징입니다.',
 '- 원문 길이와 출처 컬럼은 모델에서 제외했지만, 편집된 보이스피싱의 위험표현 밀도가 높다는 편향은 남을 수 있습니다.',
 '- K-means 군집은 사전 정답 없이 만든 유사 패턴이며 정상·사기 분류 정답이 아닙니다.',
 '- 군집 이름은 특징 프로필과 대표 원문을 확인한 뒤 사람이 부여해야 합니다.',
 '- 현재 결과는 두 공개 코퍼스 분석이며 실제 서비스 일반화 성능은 외부 데이터로 검증해야 합니다.'
]
(REPORT_ROOT/'03_3_interpretable_feature_report.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'03-3_v1','seed':SEED,'dataset_root':str(DATASET_ROOT),'feature_count':len(feature_cols),
          'best_supervised_model':best_name,'test_metrics':test_metrics,'mixed_kmeans_k':best_k,
          'mixed_kmeans_ari':ari,'mixed_kmeans_nmi':nmi,'fraud_tactic_k':best_fraud_k,
          'group_leakage_check':True,'excluded_features':['text_length','source_group','financial_topic','original_split','file_name']}
(REPORT_ROOT/'03_3_run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
assert feature_df.groupby('group_id').ml_split.nunique().max()==1
assert len(list(MODEL_ROOT.glob('*.joblib')))==3
print('03-3 정상 완료:',OUTPUT_ROOT)